# P14b — ZORC LangGraph Agent Demo

**ZORC Pipeline — Zip-code Of RNAs that Condense**  
MoschouLab / IMBB-FORTH / ERC PLANTEX

This notebook demonstrates the **P14b LangGraph agent**: a multi-step workflow that integrates  
ZORC ML predictions (RandomForest, P9d) with P-body literature (ChromaDB RAG, P14a) and  
generates structured gene-level reports via Claude claude-sonnet-4-6.

## Workflow graph

```
START
  └─► get_prediction    (SQLite lookup → FastAPI fallback)
        │
        ├─ prob_pos > 0.8 ──► retrieve_literature  (ChromaDB RAG)
        │                            │
        │                            ▼
        └─ prob_pos ≤ 0.8 ──► generate_report  (Claude claude-sonnet-4-6)
                                     │
                                     ▼
                                    END
```

## Test genes

| Gene | Description | prob_pos | Expected route |
|------|-------------|----------|----------------|
| AT5G47010 | UPF1 / LBA1 — RNA helicase, known P-body component | 0.930 | `retrieve_literature → generate_report` |
| AT1G01470 | NAC001 TF — negative control | 0.252 | `generate_report` (direct) |
| AT3G22270 | PAT1 — P-body scaffold | 0.925 | `retrieve_literature → generate_report` |

## Prerequisites

```bash
conda activate zorc_pipeline
export ANTHROPIC_API_KEY=sk-ant-...   # required for LLM report generation
# FastAPI optional (SQLite fallback active when not running)
uvicorn api.main:app --port 8000 &
```

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display

# Ensure repo root is on the path
REPO_ROOT = Path().resolve().parent  # notebooks run from agent/ dir
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Verify key packages
import langgraph, anthropic, langchain_chroma
print(f"LangGraph  : {langgraph.__version__}")
print(f"Anthropic  : {anthropic.__version__}")

api_key_set = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"API key set: {'✓' if api_key_set else '✗  (set ANTHROPIC_API_KEY for LLM reports)'}")

## 2. Build the agent and inspect the graph

In [ ]:
from agent.zorc_agent import build_agent

agent = build_agent()
print("Graph nodes:", list(agent.get_graph().nodes.keys()))
print("Graph edges:")
for src, dst, _ in agent.get_graph().edges:
    print(f"  {src} → {dst}")

In [ ]:
# Draw the graph (requires pygraphviz or mermaid)
try:
    from IPython.display import Image
    img = agent.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    print(f"Graph visualisation not available: {e}")
    print("Install pygraphviz for PNG rendering.")

## 3. Run: AT5G47010 — UPF1/LBA1 (high-confidence P-body component)

Expected route: `get_prediction (prob=0.930)` → `retrieve_literature` → `generate_report`

In [ ]:
state_pos = agent.invoke({"gene_id": "AT5G47010"})

print(f"Gene       : {state_pos['gene_id']}")
print(f"prob_pos   : {state_pos['prob_pos']:.4f}")
print(f"Prediction : {state_pos['prediction']}")
print(f"Confidence : {state_pos['confidence']}")
print(f"Source     : {state_pos['lookup_source']}")
print(f"Lit chunks : {len(state_pos.get('literature') or [])}")

In [ ]:
# Top SHAP features
shap = state_pos.get("shap_features") or {}
print("Top SHAP contributions:")
for feat, val in sorted(shap.items(), key=lambda kv: abs(kv[1]), reverse=True):
    bar = "█" * int(abs(val) * 400) + ("▊" if val > 0 else "▌")
    sign = "+" if val >= 0 else ""
    print(f"  {feat:<22} {sign}{val:+.4f}  {bar}")

In [ ]:
# Retrieved literature chunks
lit = state_pos.get("literature") or []
print(f"Retrieved {len(lit)} literature chunks:\n")
for h in lit[:3]:
    print(f"[{h['rank']}] {h['source']}  page {h['page'] + 1}  score={h['score']:.3f}")
    print(f"    {h['text'][:300].strip()}…\n")

In [ ]:
# Full structured report
display(Markdown(state_pos["report"]))

## 4. Run: AT1G01470 — NAC001 TF (negative control)

Expected route: `get_prediction (prob=0.252)` → `generate_report` (no literature retrieval)

In [ ]:
state_neg = agent.invoke({"gene_id": "AT1G01470"})

print(f"Gene       : {state_neg['gene_id']}")
print(f"prob_pos   : {state_neg['prob_pos']:.4f}")
print(f"Prediction : {state_neg['prediction']}")
print(f"Confidence : {state_neg['confidence']}")
print(f"Literature : {len(state_neg.get('literature') or [])} chunks  (0 expected for neg control)")

In [ ]:
display(Markdown(state_neg["report"]))

## 5. Run: AT3G22270 — PAT1 (P-body scaffold, high-confidence positive)

PAT1 is a known P-body scaffold protein. Expected prob_pos ≈ 0.925.

In [ ]:
state_pat1 = agent.invoke({"gene_id": "AT3G22270"})

print(f"Gene       : {state_pat1['gene_id']}")
print(f"prob_pos   : {state_pat1['prob_pos']:.4f}")
print(f"Prediction : {state_pat1['prediction']}")
print(f"Confidence : {state_pat1['confidence']}")
print(f"Lit chunks : {len(state_pat1.get('literature') or [])}")

In [ ]:
display(Markdown(state_pat1["report"]))

## 6. Batch comparison

In [ ]:
import pandas as pd

genes = ["AT5G47010", "AT3G22270", "AT1G01470"]
rows = []
for g in genes:
    s = agent.invoke({"gene_id": g})
    rows.append({
        "gene_id":    s["gene_id"],
        "prob_pos":   s["prob_pos"],
        "prediction": s["prediction"],
        "confidence": s["confidence"],
        "lit_chunks": len(s.get("literature") or []),
        "route":      "lit+report" if s.get("literature") else "report_only",
    })

df = pd.DataFrame(rows)
df.style.background_gradient(subset=["prob_pos"], cmap="RdYlGn")

## 7. CLI usage

The agent is also available as a command-line tool:

```bash
# Single gene
python agent/run_agent.py AT5G47010

# Multiple genes
python agent/run_agent.py AT5G47010 AT3G22270 AT1G01470

# Machine-readable JSON output
python agent/run_agent.py AT5G47010 AT1G01470 --json

# Skip LLM (test prediction + RAG only)
python agent/run_agent.py AT5G47010 --no-llm
```

## Summary

| Component | Implementation |
|-----------|----------------|
| Framework | LangGraph 1.1.9 — `StateGraph` with `TypedDict` state |
| Prediction | FastAPI `/lookup/{gene_id}` → SQLite fallback |
| Literature | ChromaDB + `all-MiniLM-L6-v2` (1,244 chunks, 10 PDFs) |
| LLM | Claude claude-sonnet-4-6 via Anthropic SDK |
| Routing | Conditional edge: prob_pos > 0.8 → RAG, else → direct LLM |
| Output | Structured markdown: Prediction + Literature + Interpretation |